# SegFormer Fine-Tuning (B0 & B2) on CFP Façade Dataset

Trains **SegFormer-B0** and **SegFormer-B2** on the CFP dataset (10 classes: background + 9 façade classes).

### Features:
- Class-weighted CrossEntropyLoss (handles heavy class imbalance)
- Mixed precision (`torch.cuda.amp.autocast`) for fast T4 training
- Saves `checkpoint_latest.pt` every epoch (safeguards against Kaggle session limits)
- Saves `best_model.pt` based on validation mIoU (excluding background)
- Reports **dataset-level mIoU** with (10 classes) and without background (9 classes)
- Automatic test evaluation and comparison summary table

In [1]:
!pip install -q transformers accelerate pillow matplotlib numpy

In [2]:
import os, sys, time, json, random, math
from pathlib import Path
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as T
from transformers import SegformerForSemanticSegmentation, get_cosine_schedule_with_warmup

# ── Device ──────────────────────────────────────────────────────────
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Using device: cuda
GPU: Tesla T4


## Configuration & Dataset Paths

In [3]:
# ── Kaggle Dataset Paths ─────────────────────────────────────────────
TRAIN_DIR = Path("/kaggle/input/datasets/baidarsamir/cfp-train/train")
VAL_DIR = Path("/kaggle/input/datasets/baidarsamir/cfp-val/val")
TEST_DIR = Path("/kaggle/input/datasets/baidarsamir/cfp-test/test")

# Fallback resolution if dataset name differs slightly
def resolve_split_path(preferred_path, keyword):
    if preferred_path.exists():
        return preferred_path
    candidates = list(Path("/kaggle/input").glob(f"**/{keyword}"))
    for c in candidates:
        if (c / "images").exists():
            print(f"Resolved {keyword} -> {c}")
            return c
    raise FileNotFoundError(f"Could not locate {keyword} images under /kaggle/input")

TRAIN_DIR = resolve_split_path(TRAIN_DIR, "train")
VAL_DIR = resolve_split_path(VAL_DIR, "val")
TEST_DIR = resolve_split_path(TEST_DIR, "test")

OUTPUT_BASE = Path("/kaggle/working/results/training")
OUTPUT_BASE.mkdir(parents=True, exist_ok=True)

# ── Class definitions ────────────────────────────────────────────────
NUM_CLASSES = 10
CLASS_NAMES = {
    0: "background", 1: "building", 2: "window", 3: "sky",
    4: "roof", 5: "door", 6: "tree", 7: "people", 8: "car", 9: "sign"
}
CLASS_COLORS_RGB = {
    0: (0, 0, 0), 1: (70, 70, 70), 2: (250, 170, 30), 3: (70, 130, 180),
    4: (0, 60, 100), 5: (153, 153, 153), 6: (107, 142, 35), 7: (255, 0, 0),
    8: (0, 0, 142), 9: (220, 220, 0)
}

print(f"Train images: {len(list((TRAIN_DIR / 'images').glob('*.png')))}")
print(f"Val images:   {len(list((VAL_DIR / 'images').glob('*.png')))}")
print(f"Test images:  {len(list((TEST_DIR / 'images').glob('*.png')))}")

Train images: 481
Val images:   60
Test images:  61


## Dataset Definition & Class Weights

In [4]:
class FacadeDataset(Dataset):
    def __init__(self, root: Path, split: str = "train", crop_size: int = 512, augment: bool = True):
        self.root = root
        self.crop_size = crop_size
        self.augment = augment and (split == "train")

        self.image_paths = sorted((self.root / "images").glob("*.png"))
        self.mask_paths = sorted((self.root / "masks").glob("*.png"))
        assert len(self.image_paths) == len(self.mask_paths), f"Mismatch in {root}"

        self.normalize = T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img = Image.open(self.image_paths[idx]).convert("RGB")
        msk = Image.open(self.mask_paths[idx])

        if self.augment:
            img, msk = self._augment(img, msk)
        else:
            img, msk = self._center_crop_or_pad(img, msk)

        img = T.ToTensor()(img)
        img = self.normalize(img)
        msk = torch.from_numpy(np.array(msk, dtype=np.int64))
        return {"pixel_values": img, "labels": msk}

    def _augment(self, img: Image.Image, msk: Image.Image):
        img, msk = self._random_crop(img, msk, self.crop_size)
        if random.random() > 0.5:
            img = img.transpose(Image.FLIP_LEFT_RIGHT)
            msk = msk.transpose(Image.FLIP_LEFT_RIGHT)
        img = T.ColorJitter(brightness=0.25, contrast=0.25, saturation=0.25)(img)
        return img, msk

    def _random_crop(self, img, msk, size):
        w, h = img.size
        if w < size or h < size:
            img, msk = self._pad(img, msk, size)
            w, h = img.size
        x = random.randint(0, w - size)
        y = random.randint(0, h - size)
        return img.crop((x, y, x + size, y + size)), msk.crop((x, y, x + size, y + size))

    def _center_crop_or_pad(self, img, msk):
        w, h = img.size
        size = self.crop_size
        if w < size or h < size:
            img, msk = self._pad(img, msk, size)
            w, h = img.size
        x = (w - size) // 2
        y = (h - size) // 2
        return img.crop((x, y, x + size, y + size)), msk.crop((x, y, x + size, y + size))

    def _pad(self, img, msk, size):
        w, h = img.size
        pw = max(0, size - w)
        ph = max(0, size - h)
        if pw > 0 or ph > 0:
            img = Image.fromarray(np.pad(np.array(img), ((0, ph), (0, pw), (0, 0)), mode="reflect"))
            msk = Image.fromarray(np.pad(np.array(msk), ((0, ph), (0, pw)), mode="constant", constant_values=0))
        return img, msk

def compute_class_weights(mask_dir: Path, num_classes: int = 10):
    counts = np.zeros(num_classes, dtype=np.float64)
    for mp in sorted(mask_dir.glob("*.png")):
        m = np.array(Image.open(mp))
        for c in range(num_classes):
            counts[c] += (m == c).sum()
    freq = counts / (counts.sum() + 1e-8)
    weights = 1.0 / (freq + 1e-8)
    cap = 10.0 * np.median(weights)
    weights = np.minimum(weights, cap)
    weights /= weights.sum() / num_classes
    return torch.tensor(weights, dtype=torch.float32)

CLASS_WEIGHTS = compute_class_weights(TRAIN_DIR / "masks", NUM_CLASSES)
print("Class weights (0-9):", [round(float(w), 2) for w in CLASS_WEIGHTS])

Class weights (0-9): [0.1, 0.03, 0.13, 0.08, 1.84, 0.59, 0.29, 4.42, 0.93, 1.6]


## Training & Evaluation Engine

In [5]:
def compute_dataset_iou(total_intersection, total_union):
    ious = {}
    v10, v9 = [], []
    for c in range(NUM_CLASSES):
        name = CLASS_NAMES[c]
        u = total_union[c]
        if u > 0:
            val = float(total_intersection[c] / u)
            ious[name] = val
            v10.append(val)
            if c > 0:
                v9.append(val)
        else:
            ious[name] = None
    return ious, float(np.mean(v10)), float(np.mean(v9))

@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    total_intersection = np.zeros(NUM_CLASSES, dtype=np.float64)
    total_union = np.zeros(NUM_CLASSES, dtype=np.float64)
    total_loss = 0.0
    criterion = nn.CrossEntropyLoss()

    for batch in loader:
        images = batch["pixel_values"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(pixel_values=images)
        logits = F.interpolate(outputs.logits, size=labels.shape[-2:], mode="bilinear", align_corners=False)
        loss = criterion(logits, labels)
        total_loss += loss.item() * images.size(0)

        preds = logits.argmax(dim=1).cpu().numpy()
        gts = labels.cpu().numpy()

        for c in range(NUM_CLASSES):
            p_c = preds == c
            g_c = gts == c
            total_intersection[c] += np.logical_and(p_c, g_c).sum()
            total_union[c] += np.logical_or(p_c, g_c).sum()

    avg_loss = total_loss / len(loader.dataset)
    ious, miou_10, miou_9 = compute_dataset_iou(total_intersection, total_union)
    return avg_loss, ious, miou_10, miou_9

def run_training(
    model_variant="b0",
    epochs=35,
    batch_size=8,
    lr=6e-5,
    weight_decay=0.01,
    resume=True,
):
    out_dir = OUTPUT_BASE / f"segformer_{model_variant}"
    out_dir.mkdir(parents=True, exist_ok=True)

    print(f"\n{'='*60}")
    print(f"Starting SegFormer-{model_variant.upper()} Training ({epochs} epochs, batch_size={batch_size})")
    print(f"{'='*60}")

    train_ds = FacadeDataset(TRAIN_DIR, split="train", crop_size=512, augment=True)
    val_ds = FacadeDataset(VAL_DIR, split="val", crop_size=512, augment=False)
    test_ds = FacadeDataset(TEST_DIR, split="test", crop_size=512, augment=False)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True, drop_last=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)

    pretrained_id = f"nvidia/segformer-{model_variant}-finetuned-ade-512-512"
    model = SegformerForSemanticSegmentation.from_pretrained(
        pretrained_id,
        num_labels=NUM_CLASSES,
        id2label={i: n for i, n in CLASS_NAMES.items()},
        label2id={n: i for i, n in CLASS_NAMES.items()},
        ignore_mismatched_sizes=True,
    ).to(DEVICE)

    criterion = nn.CrossEntropyLoss(weight=CLASS_WEIGHTS.to(DEVICE))
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

    total_steps = len(train_loader) * epochs
    scheduler = get_cosine_schedule_with_warmup(optimizer, num_warmup_steps=int(0.05 * total_steps), num_training_steps=total_steps)
    scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE.type == "cuda"))

    start_epoch = 1
    best_miou_no_bg = 0.0
    history = []

    latest_ckpt = out_dir / "checkpoint_latest.pt"
    if resume and latest_ckpt.exists():
        print(f"Found checkpoint: {latest_ckpt}. Resuming...")
        ck = torch.load(latest_ckpt, map_location=DEVICE)
        model.load_state_dict(ck["model_state_dict"])
        optimizer.load_state_dict(ck["optimizer_state_dict"])
        scheduler.load_state_dict(ck["scheduler_state_dict"])
        start_epoch = ck["epoch"] + 1
        best_miou_no_bg = ck["best_miou"]
        history = ck.get("history", [])
        print(f"Resumed at epoch {start_epoch} (Best mIoU no bg: {best_miou_no_bg*100:.2f}%)")

    for epoch in range(start_epoch, epochs + 1):
        model.train()
        train_loss = 0.0
        t0 = time.time()

        for batch in train_loader:
            images = batch["pixel_values"].to(DEVICE)
            labels = batch["labels"].to(DEVICE)

            optimizer.zero_grad()
            with torch.cuda.amp.autocast(enabled=(DEVICE.type == "cuda")):
                outputs = model(pixel_values=images)
                logits = F.interpolate(outputs.logits, size=labels.shape[-2:], mode="bilinear", align_corners=False)
                loss = criterion(logits, labels)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()

            train_loss += loss.item() * images.size(0)

        train_loss /= len(train_loader.dataset)
        val_loss, val_ious, val_m10, val_m9 = evaluate(model, val_loader, DEVICE)
        elapsed = time.time() - t0

        print(f"Epoch [{epoch:02d}/{epochs:02d}] {elapsed:4.1f}s | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val mIoU (10): {val_m10*100:5.2f}% | Val mIoU (no bg): {val_m9*100:5.2f}%")

        record = {"epoch": epoch, "train_loss": train_loss, "val_loss": val_loss, "val_miou_10": val_m10, "val_miou_9_no_bg": val_m9, "val_ious": val_ious}
        history.append(record)

        # Save periodic checkpoint
        torch.save({"epoch": epoch, "model_state_dict": model.state_dict(), "optimizer_state_dict": optimizer.state_dict(), "scheduler_state_dict": scheduler.state_dict(), "best_miou": best_miou_no_bg, "history": history}, latest_ckpt)

        # Save best model
        if val_m9 > best_miou_no_bg:
            best_miou_no_bg = val_m9
            torch.save(model.state_dict(), out_dir / "best_model.pt")
            print(f"  ★ New best model! (Val mIoU no bg: {best_miou_no_bg*100:.2f}%)")

        with open(out_dir / "train_history.json", "w") as f:
            json.dump(history, f, indent=2)

    # Test set evaluation with best weights
    print(f"\nEvaluating SegFormer-{model_variant.upper()} on Test Set...")
    best_weights = out_dir / "best_model.pt"
    if best_weights.exists():
        model.load_state_dict(torch.load(best_weights, map_location=DEVICE))

    test_loss, test_ious, test_m10, test_m9 = evaluate(model, test_loader, DEVICE)
    print(f"Test mIoU (10 classes): {test_m10*100:.2f}%")
    print(f"Test mIoU (9 classes, no bg): {test_m9*100:.2f}%")

    test_metrics = {"model": f"SegFormer-{model_variant.upper()}", "test_loss": test_loss, "miou_10": test_m10, "miou_9_no_bg": test_m9, "per_class_iou": test_ious}
    with open(out_dir / "test_metrics.json", "w") as f:
        json.dump(test_metrics, f, indent=2)

    return test_metrics, history

## 1. Train SegFormer-B0

In [6]:
# SegFormer-B0: lightweight (~3.8M params), batch_size=8 or 16, ~15-20 min on T4
b0_metrics, b0_history = run_training(
    model_variant="b0",
    epochs=35,
    batch_size=8,
    lr=6e-5,
    resume=True,
)


Starting SegFormer-B0 Training (35 epochs, batch_size=8)


config.json:   0%|          | 0.00/6.88k [00:00<?, ?B/s]

[transformers] You passed `num_labels=10` which is incompatible to the `id2label` map of length `150`.


model.safetensors: reconstructing file:   0%|          |  0.00B / 15.0MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/208 [00:00<?, ?it/s]

[transformers] SegformerForSemanticSegmentation LOAD REPORT from: nvidia/segformer-b0-finetuned-ade-512-512
Key                           | Status   |                                                                                                      
------------------------------+----------+------------------------------------------------------------------------------------------------------
decode_head.classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([150, 256, 1, 1]) vs model:torch.Size([10, 256, 1, 1])
decode_head.classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([150]) vs model:torch.Size([10])                      

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.
/tmp/ipykernel_49/2770379362.py:84: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(

Epoch [01/35] 30.4s | Train Loss: 2.2274 | Val Loss: 2.1007 | Val mIoU (10): 23.66% | Val mIoU (no bg): 23.24%
  ★ New best model! (Val mIoU no bg: 23.24%)
Epoch [02/35] 13.5s | Train Loss: 1.8378 | Val Loss: 1.7662 | Val mIoU (10): 34.13% | Val mIoU (no bg): 30.85%
  ★ New best model! (Val mIoU no bg: 30.85%)
Epoch [03/35] 14.1s | Train Loss: 1.3698 | Val Loss: 1.6127 | Val mIoU (10): 38.39% | Val mIoU (no bg): 35.25%
  ★ New best model! (Val mIoU no bg: 35.25%)
Epoch [04/35] 13.9s | Train Loss: 1.0904 | Val Loss: 1.4834 | Val mIoU (10): 39.39% | Val mIoU (no bg): 36.14%
  ★ New best model! (Val mIoU no bg: 36.14%)
Epoch [05/35] 14.1s | Train Loss: 0.9148 | Val Loss: 1.3660 | Val mIoU (10): 41.95% | Val mIoU (no bg): 39.12%
  ★ New best model! (Val mIoU no bg: 39.12%)
Epoch [06/35] 14.3s | Train Loss: 0.7951 | Val Loss: 1.2712 | Val mIoU (10): 41.05% | Val mIoU (no bg): 38.30%
Epoch [07/35] 14.0s | Train Loss: 0.7202 | Val Loss: 1.1715 | Val mIoU (10): 43.73% | Val mIoU (no bg): 40.98

## 2. Train SegFormer-B2

In [7]:
# SegFormer-B2: higher capacity (~27.5M params), batch_size=4 or 8, ~35-45 min on T4
b2_metrics, b2_history = run_training(
    model_variant="b2",
    epochs=35,
    batch_size=4,
    lr=6e-5,
    resume=True,
)


Starting SegFormer-B2 Training (35 epochs, batch_size=4)


config.json:   0%|          | 0.00/6.88k [00:00<?, ?B/s]

[transformers] You passed `num_labels=10` which is incompatible to the `id2label` map of length `150`.


pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  110MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/380 [00:00<?, ?it/s]

[transformers] SegformerForSemanticSegmentation LOAD REPORT from: nvidia/segformer-b2-finetuned-ade-512-512
Key                           | Status   |                                                                                                      
------------------------------+----------+------------------------------------------------------------------------------------------------------
decode_head.classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([150]) vs model:torch.Size([10])                      
decode_head.classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([150, 768, 1, 1]) vs model:torch.Size([10, 768, 1, 1])

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.
/tmp/ipykernel_49/2770379362.py:84: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(

model.safetensors: reconstructing file:   0%|          |  0.00B /  110MB            

model.safetensors: downloading bytes:           |  0.00B            

/tmp/ipykernel_49/2770379362.py:112: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(DEVICE.type == "cuda")):


Epoch [01/35] 40.1s | Train Loss: 2.1212 | Val Loss: 1.7345 | Val mIoU (10): 40.99% | Val mIoU (no bg): 38.60%
  ★ New best model! (Val mIoU no bg: 38.60%)
Epoch [02/35] 38.9s | Train Loss: 1.2491 | Val Loss: 1.3429 | Val mIoU (10): 40.09% | Val mIoU (no bg): 37.11%
Epoch [03/35] 39.1s | Train Loss: 0.7565 | Val Loss: 1.0712 | Val mIoU (10): 46.47% | Val mIoU (no bg): 43.74%
  ★ New best model! (Val mIoU no bg: 43.74%)
Epoch [04/35] 39.4s | Train Loss: 0.6013 | Val Loss: 0.9326 | Val mIoU (10): 50.02% | Val mIoU (no bg): 48.11%
  ★ New best model! (Val mIoU no bg: 48.11%)
Epoch [05/35] 39.1s | Train Loss: 0.4951 | Val Loss: 0.7640 | Val mIoU (10): 51.01% | Val mIoU (no bg): 48.76%
  ★ New best model! (Val mIoU no bg: 48.76%)
Epoch [06/35] 38.8s | Train Loss: 0.4553 | Val Loss: 0.7015 | Val mIoU (10): 54.09% | Val mIoU (no bg): 52.04%
  ★ New best model! (Val mIoU no bg: 52.04%)
Epoch [07/35] 38.8s | Train Loss: 0.4156 | Val Loss: 0.6522 | Val mIoU (10): 54.16% | Val mIoU (no bg): 51.96

## 3. Compare Models & Visualize Results

In [8]:
import pandas as pd

# Per-class IoU summary table
table_data = []
for c in range(NUM_CLASSES):
    name = CLASS_NAMES[c]
    b0_v = b0_metrics["per_class_iou"].get(name)
    b2_v = b2_metrics["per_class_iou"].get(name)
    table_data.append({
        "Class": name,
        "SegFormer-B0": f"{b0_v*100:5.2f}%" if b0_v is not None else "N/A",
        "SegFormer-B2": f"{b2_v*100:5.2f}%" if b2_v is not None else "N/A",
    })

df = pd.DataFrame(table_data)
print(df.to_string(index=False))
print("-" * 40)
print(f"mIoU 10 classes    | B0: {b0_metrics['miou_10']*100:5.2f}% | B2: {b2_metrics['miou_10']*100:5.2f}%")
print(f"mIoU (no bg, 9cls) | B0: {b0_metrics['miou_9_no_bg']*100:5.2f}% | B2: {b2_metrics['miou_9_no_bg']*100:5.2f}%")

# Save comparison summary
summary = {
    "b0": b0_metrics,
    "b2": b2_metrics,
}
with open(OUTPUT_BASE / "models_comparison_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

     Class SegFormer-B0 SegFormer-B2
background       64.55%       68.70%
  building       68.73%       77.47%
    window       57.96%       64.48%
       sky       89.64%       91.88%
      roof       30.00%       39.77%
      door       39.36%       49.50%
      tree       47.35%       46.63%
    people       18.42%       36.57%
       car       57.55%       69.55%
      sign       18.20%       22.04%
----------------------------------------
mIoU 10 classes    | B0: 49.18% | B2: 56.66%
mIoU (no bg, 9cls) | B0: 47.47% | B2: 55.32%


## 4. Package Artifacts for Download

In [10]:
import shutil, os
os.makedirs("/kaggle/working/models_out", exist_ok=True)
for m in ["b0", "b2"]:
    shutil.copy(f"/kaggle/working/results/training/segformer_{m}/best_model.pt",
                f"/kaggle/working/models_out/segformer_{m}_best.pt")
shutil.make_archive("/kaggle/working/best_models", "zip", "/kaggle/working/models_out")
print(os.listdir("/kaggle/working"))

['results', 'training_results.zip', '.virtual_documents', 'best_models.zip', 'models_out']


In [12]:
import os, glob, json, numpy as np, torch, torch.nn.functional as F
from PIL import Image
from transformers import SegformerForSemanticSegmentation

TEST = "/kaggle/input/datasets/baidarsamir/cfp-test/test"
IMG_DIR, MSK_DIR = f"{TEST}/images", f"{TEST}/masks"
NAMES = ["background","building","window","sky","roof","door","tree","people","car","sign"]
dev = torch.device("cuda")
mean = torch.tensor([0.485,0.456,0.406],device=dev).view(1,3,1,1)
std  = torch.tensor([0.229,0.224,0.225],device=dev).view(1,3,1,1)

def load(tag, hf):
    m = SegformerForSemanticSegmentation.from_pretrained(hf, num_labels=10, ignore_mismatched_sizes=True)
    sd = torch.load(f"/kaggle/working/results/training/segformer_{tag}/best_model.pt", map_location="cpu")
    m.load_state_dict(sd)
    return m.to(dev).eval()

@torch.no_grad()
def evaluate(model):
    conf = np.zeros((10,10), dtype=np.int64)
    ips = sorted(glob.glob(f"{IMG_DIR}/*.png"))
    mps = sorted(glob.glob(f"{MSK_DIR}/*.png"))
    assert len(ips) == len(mps) == 61, (len(ips), len(mps))
    for ip, mp in zip(ips, mps):
        gt = np.array(Image.open(mp))
        gt = gt[..., 0] if gt.ndim == 3 else gt
        x = torch.from_numpy(np.array(Image.open(ip).convert("RGB"))).permute(2,0,1)[None].float().div(255).to(dev)
        x = (x - mean) / std
        h, w = x.shape[-2:]
        ph, pw = (32 - h % 32) % 32, (32 - w % 32) % 32
        x = F.pad(x, (0, pw, 0, ph), mode="reflect")
        lg = model(pixel_values=x).logits
        lg = F.interpolate(lg, size=x.shape[-2:], mode="bilinear", align_corners=False)[..., :h, :w]
        pred = lg.argmax(1)[0].cpu().numpy()
        conf += np.bincount(gt.astype(np.int64).ravel()*10 + pred.ravel(), minlength=100).reshape(10,10)
    iou = np.diag(conf) / (conf.sum(0) + conf.sum(1) - np.diag(conf))
    return iou, conf

out = {}
for tag, hf in [("b0","nvidia/segformer-b0-finetuned-ade-512-512"),
                ("b2","nvidia/segformer-b2-finetuned-ade-512-512")]:
    iou, conf = evaluate(load(tag, hf))
    out[tag] = {"per_class_iou": dict(zip(NAMES, (iou*100).round(2).tolist())),
                "miou_10": round(float(iou.mean()*100), 2),
                "miou_9_no_bg": round(float(iou[1:].mean()*100), 2)}
    np.save(f"/kaggle/working/confusion_{tag}_full.npy", conf)
    print(tag, out[tag])
json.dump(out, open("/kaggle/working/full_image_results.json","w"), indent=2)

[transformers] You passed `num_labels=10` which is incompatible to the `id2label` map of length `150`.


Loading weights:   0%|          | 0/208 [00:00<?, ?it/s]

[transformers] SegformerForSemanticSegmentation LOAD REPORT from: nvidia/segformer-b0-finetuned-ade-512-512
Key                           | Status   |                                                                                                      
------------------------------+----------+------------------------------------------------------------------------------------------------------
decode_head.classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([150, 256, 1, 1]) vs model:torch.Size([10, 256, 1, 1])
decode_head.classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([150]) vs model:torch.Size([10])                      

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.
[transformers] You passed `num_labels=10` which is incompatible to the `id2label` map of length `150`.


b0 {'per_class_iou': {'background': 67.97, 'building': 66.02, 'window': 51.91, 'sky': 90.77, 'roof': 26.48, 'door': 30.9, 'tree': 49.47, 'people': 21.42, 'car': 60.62, 'sign': 17.84}, 'miou_10': 48.34, 'miou_9_no_bg': 46.16}


Loading weights:   0%|          | 0/380 [00:00<?, ?it/s]

[transformers] SegformerForSemanticSegmentation LOAD REPORT from: nvidia/segformer-b2-finetuned-ade-512-512
Key                           | Status   |                                                                                                      
------------------------------+----------+------------------------------------------------------------------------------------------------------
decode_head.classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([150]) vs model:torch.Size([10])                      
decode_head.classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([150, 768, 1, 1]) vs model:torch.Size([10, 768, 1, 1])

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


b2 {'per_class_iou': {'background': 69.27, 'building': 73.62, 'window': 58.48, 'sky': 92.33, 'roof': 40.49, 'door': 40.37, 'tree': 49.82, 'people': 39.89, 'car': 75.51, 'sign': 22.45}, 'miou_10': 56.22, 'miou_9_no_bg': 54.77}


In [13]:
import json, torch, numpy as np
from pathlib import Path
from PIL import Image
from transformers import SegformerForSemanticSegmentation
import torch.nn.functional as F

CKPT = "/kaggle/working/results/training/segformer_b2/best_model.pt"
TEST_DIR = Path("/kaggle/input/datasets/baidarsamir/cfp-test/test")
IMG_DIR, MSK_DIR = TEST_DIR / "images", TEST_DIR / "masks"
OUT_JSON = "/kaggle/working/wwr_predictions.json"
NUM_CLASSES = 10
MIN_GROSS_PX = 1000

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = SegformerForSemanticSegmentation.from_pretrained(
    "nvidia/segformer-b2-finetuned-ade-512-512",
    num_labels=NUM_CLASSES, ignore_mismatched_sizes=True)
model.load_state_dict(torch.load(CKPT, map_location="cpu"))
model.to(device).eval()

MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
STD = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

def wwr_from_mask(m):
    win, bld, door = int((m == 2).sum()), int((m == 1).sum()), int((m == 5).sum())
    gross = win + bld + door
    return (win / gross if gross >= MIN_GROSS_PX else None), gross

img_paths = sorted(IMG_DIR.glob("*.png"))
mask_paths = sorted(MSK_DIR.glob("*.png"))
assert len(img_paths) == len(mask_paths) == 61, (len(img_paths), len(mask_paths))

records = []
for img_p, msk_p in zip(img_paths, mask_paths):
    gt = np.array(Image.open(msk_p))
    gt = gt[..., 0] if gt.ndim == 3 else gt
    gt_wwr, gt_gross = wwr_from_mask(gt)

    img = np.array(Image.open(img_p).convert("RGB"))
    t = (torch.from_numpy(img).float().permute(2, 0, 1) / 255.0 - MEAN) / STD
    inp = t.unsqueeze(0)
    h, w = inp.shape[-2:]
    ph, pw = (32 - h % 32) % 32, (32 - w % 32) % 32
    x = F.pad(inp, (0, pw, 0, ph), mode="reflect").to(device)
    with torch.no_grad():
        logits = model(pixel_values=x).logits
    up = F.interpolate(logits, size=x.shape[-2:], mode="bilinear", align_corners=False)[..., :h, :w]
    pred = up.argmax(1).squeeze(0).cpu().numpy()
    pred_wwr, pred_gross = wwr_from_mask(pred)

    records.append({"stem": img_p.stem, "gt_wwr": gt_wwr, "pred_wwr": pred_wwr,
                    "gt_gross_px": gt_gross, "pred_gross_px": pred_gross})

json.dump(records, open(OUT_JSON, "w"), indent=2)
ok = [r for r in records if r["gt_wwr"] is not None and r["pred_wwr"] is not None]
print(f"Saved {len(records)} records, {len(ok)} usable")
print("mean GT WWR  :", round(float(np.mean([r['gt_wwr'] for r in ok])), 4))
print("mean pred WWR:", round(float(np.mean([r['pred_wwr'] for r in ok])), 4))

[transformers] You passed `num_labels=10` which is incompatible to the `id2label` map of length `150`.


Loading weights:   0%|          | 0/380 [00:00<?, ?it/s]

[transformers] SegformerForSemanticSegmentation LOAD REPORT from: nvidia/segformer-b2-finetuned-ade-512-512
Key                           | Status   |                                                                                                      
------------------------------+----------+------------------------------------------------------------------------------------------------------
decode_head.classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([150]) vs model:torch.Size([10])                      
decode_head.classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([150, 768, 1, 1]) vs model:torch.Size([10, 768, 1, 1])

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


Saved 61 records, 61 usable
mean GT WWR  : 0.1655
mean pred WWR: 0.2358
